In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV = configs.get("env", "dev")
OPTIMIZE_FULL = configs.get("optimize_full", 'False').lower() == "true"
SCHEMAS = configs.get("schemas", ['bronze', 'silver', 'gold'])

CATALOG = f"sl_{ENV}"

print(ENV, OPTIMIZE_FULL, CATALOG, SCHEMAS)

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
def get_table_list(schemas: str):
    tables = list()

    for schema in schemas:
        df = (spark.sql(f"""SELECT table_name
                            FROM information_schema.tables
                            WHERE table_schema = '{schema}'
                            AND table_type = 'MANAGED'
                            AND data_source_format = 'DELTA'""")
              .collect())
        tables += list(map(lambda row: f"{schema}.{row['table_name']}", df))
    
    return tables


In [0]:
def optimize_table(table: list, optimize_full: bool):
    spark.sql(f"OPTIMIZE {table} {'FULL' if optimize_full else ''} ")
    spark.sql(f"ANALYZE TABLE {CATALOG}.{table} COMPUTE STATISTICS")
    print(f"table {table} optimized")

In [0]:
tables = get_table_list(SCHEMAS)

In [0]:
for table in tables:
    optimize_table(table, OPTIMIZE_FULL)